# Лабораторная работа №6

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "Helsinki-NLP/opus-mt-en-ru"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


C:\Users\magofrays\PycharmProjects\lb6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 86006.23it/s]


In [17]:
from datasets import load_dataset
ds_postpunk = load_dataset("AlekseyCalvin/song_lyrics_Ru2En_PostSoviet_alt_anthems", split="train")
print("Колонки:", ds_postpunk.column_names)

ds_postpunk = ds_postpunk.map(
    lambda x: {
        "ru": str(x["input"]).strip(),
        "en": str(x["accepted"]).strip()
    },
    remove_columns=["instruction", "input", "accepted", "rejected"]
)
ds_postpunk = ds_postpunk.filter(lambda x: len(x["en"]) > 0 and len(x["ru"]) > 0)
print(f"{len(ds_postpunk)} пар")

Колонки: ['instruction', 'input', 'accepted', 'rejected']
2030 пар


In [2]:
import re
def split_text(text, max_chars=200):
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) < max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence

    if current:
        chunks.append(current.strip())

    return chunks

In [3]:
def flat_map(pairs):
    all_pairs = []
    for row in pairs:
        for ru_sent, en_sent in zip(row["ru"], row["en"]):
            all_pairs.append({"ru": ru_sent, "en": en_sent})
    Dataset.from_list(all_pairs)

In [6]:
from datasets import load_dataset, Dataset
ru_en_story_pairs = load_dataset("limloop/ru_en_story_pairs", split="train")
print("Колонки:", ru_en_story_pairs.column_names)
ru_en_story_pairs = ru_en_story_pairs.map(
    lambda x: {
        "ru": (str(x["text_ru"]).strip()),
        "en": (str(x["text_en"]).strip())
    },
    remove_columns=ru_en_story_pairs.column_names
)

print(len(ru_en_story_pairs))


Колонки: ['text_ru', 'text_en', 'summary_ru', 'summary_en', 'genre']
118499


In [7]:
from datasets import load_dataset, Dataset

dataset = load_dataset("fyaronskiy/cornstack_go_ru_en", split="train", streaming=True)


cornstack_go_ru_en = dataset.take(100000)
print("to list")
cornstack_go_ru_en =  Dataset.from_list(list(cornstack_go_ru_en))
print("Колонки:", cornstack_go_ru_en.column_names)

cornstack_go_ru_en = cornstack_go_ru_en.map(
    lambda x: {
        "ru": str(x["ru_query"]).strip(),
        "en": str(x["query"]).strip()
    },
    remove_columns=cornstack_go_ru_en.column_names
)
cornstack_go_ru_en = cornstack_go_ru_en.filter(lambda x: len(x["en"]) > 0 and len(x["ru"]) > 0)
print(f"{len(cornstack_go_ru_en)} пар")

to list
Колонки: ['query', 'ru_query', 'document', 'metadata', 'negatives', 'negative_scores', 'document_score', 'document_rank']


Filter: 100%|██████████| 100000/100000 [00:00<00:00, 391096.26 examples/s]

100000 пар


In [8]:
from datasets import load_dataset
wikihow_en_ru = load_dataset("VityaVitalich/wikihow_en_ru", split="train")
print("Колонки:", wikihow_en_ru.column_names)

wikihow_en_ru = wikihow_en_ru.map(
    lambda x: {
        "ru": str(x["source"]).strip(),
        "en": str(x["target"]).strip()
    },
    remove_columns=wikihow_en_ru.column_names
)
wikihow_en_ru = wikihow_en_ru.filter(lambda x: len(x["en"]) > 0 and len(x["ru"]) > 0)
print(f"{len(wikihow_en_ru)} пар")

Колонки: ['source', 'target', 'source_sentences', 'target_sentences']
35313 пар


In [9]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
MAX_LENGTH = 128
def preprocess_function(examples):
    model_inputs = tokenizer(examples["en"], max_length=MAX_LENGTH, truncation=True)
    labels = tokenizer(examples["ru"], max_length=MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
OUTPUT_DIR = "models/opus-mt-en-ru-postpunk"
ds_postpunk_split = ds_postpunk.train_test_split(test_size=0.15, seed=42)
tokenized_postpunk = ds_postpunk_split.map(preprocess_function, batched=True)
eval_dataset_postpunk = tokenized_postpunk["test"]
training_args_postpunk = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=0.1,
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=torch.cuda.is_available()
)

trainer_poetic = Seq2SeqTrainer(
    model=model,
    args=training_args_postpunk,
    train_dataset=tokenized_postpunk["train"],
    eval_dataset=eval_dataset_postpunk,
    data_collator=data_collator,
)

print(f"Запуск дообучения на postpunk. Train: {len(tokenized_postpunk['train'])}, Val: {len(eval_dataset_postpunk)}")
trainer_poetic.train()
trainer_poetic.save_model(OUTPUT_DIR)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = "models/opus-mt-en-ru_ru_en_story_pairs"

ru_en_story_pairs_split = ru_en_story_pairs.train_test_split(test_size=0.15, seed=42)

print("Токенизация датасета...")
tokenized_ru_en_story_pairs = ru_en_story_pairs_split.map(preprocess_function, batched=True)

train_dataset = tokenized_ru_en_story_pairs["train"]
eval_dataset = tokenized_ru_en_story_pairs["test"]
print("Тренировка модели...")
training_args_ru_en_story_pairs = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=0.1,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    optim="adamw_torch",
)

trainer_ru_en = Seq2SeqTrainer(
    model=model,
    args=training_args_ru_en_story_pairs,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print(f"Запуск дообучения на ru_en_story_pairs. Train: {len(train_dataset)}, Val: {len(eval_dataset)}")
trainer_ru_en.train()
trainer_ru_en.save_model(OUTPUT_DIR)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = "models/opus-mt-en-ru_cornstack_go_ru_en"

cornstack_go_ru_en_split =cornstack_go_ru_en.train_test_split(test_size=0.15, seed=42)

print("Токенизация датасета...")
tokenized_cornstack_go_ru_en = cornstack_go_ru_en_split.map(preprocess_function, batched=True)

train_dataset = tokenized_cornstack_go_ru_en["train"]
eval_dataset = tokenized_cornstack_go_ru_en["test"]
print("Тренировка модели...")
training_args_cornstack_go_ru_en = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=0.1,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    optim="adamw_torch",
)

trainer_ru_en = Seq2SeqTrainer(
    model=model,
    args=training_args_cornstack_go_ru_en,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print(f"Запуск дообучения на ru_en_story_pairs. Train: {len(train_dataset)}, Val: {len(eval_dataset)}")
trainer_ru_en.train()
trainer_ru_en.save_model(OUTPUT_DIR)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = "models/opus-mt-en-ru_wikihow_en_ru"

wikihow_en_ru_split = wikihow_en_ru.train_test_split(test_size=0.15, seed=42)

print("Токенизация датасета...")
tokenized_wikihow_en_ru = wikihow_en_ru_split.map(preprocess_function, batched=True)

train_dataset = tokenized_wikihow_en_ru["train"]
eval_dataset = tokenized_wikihow_en_ru["test"]
print("Тренировка модели...")
training_args_wikihow_en_ru = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=0.1,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    optim="adamw_torch",
)

trainer_ru_en = Seq2SeqTrainer(
    model=model,
    args=training_args_wikihow_en_ru,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print(f"Запуск дообучения на wikihow_en_ru. Train: {len(train_dataset)}, Val: {len(eval_dataset)}")
trainer_ru_en.train()
trainer_ru_en.save_model(OUTPUT_DIR)

In [10]:
import random

def get_test_samples(dataset, n=5, seed=42):
    random.seed(seed)
    if len(dataset) < n:
        raise ValueError(f"В датасете всего {len(dataset)} примеров, запрошено {n}")
    indices = random.sample(range(len(dataset)), n)
    return [dataset[i] for i in indices]

In [11]:
import evaluate

bleu_metric = evaluate.load("sacrebleu")

def compute_bleu(prediction, reference):
    result = bleu_metric.compute(
        predictions=[prediction],
        references=[[reference]]
    )
    return result["score"]

In [13]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

MODEL_PATHS = {
    "Base": "Helsinki-NLP/opus-mt-en-ru",
    "Postpunk": "models/opus-mt-en-ru-postpunk",
    "Ru_en_story_pairs": "models/opus-mt-en-ru_ru_en_story_pairs",
    "Go_ru_en": "models/opus-mt-en-ru_cornstack_go_ru_en",
    "Wikihow": "models/opus-mt-en-ru_wikihow_en_ru",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
bleu = evaluate.load("sacrebleu")

loaded_models = {}
for name, path in MODEL_PATHS.items():
    try:
        model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
        loaded_models[name] = model
        print(f"{name} загружена")
    except Exception as e:
        print(f"Ошибка загрузки {name}: {e}")

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 51527.57it/s]


Base загружена


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 5521.80it/s]


Postpunk загружена


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 7057.05it/s]


Ru_en_story_pairs загружена


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 6349.70it/s]


Go_ru_en загружена


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 5906.68it/s]


Wikihow загружена


In [14]:
import torch

def translate_long_text(text, model, tokenizer, device):
    chunks = split_text(text)

    translated_chunks = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=4,
                max_new_tokens=256,
                no_repeat_ngram_size=3,
                repetition_penalty=1.05,
            )

        translated = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        translated_chunks.append(translated)

    return " ".join(translated_chunks)

In [30]:
test_samples = get_test_samples(wikihow_en_ru, n=10, seed=41)

ens = [s["en"] for s in test_samples]
refs = [s["ru"] for s in test_samples]

In [31]:
if 'loaded_models' not in globals() or not loaded_models:
    raise RuntimeError("Модели не загружены")

results = {}

for name, model in loaded_models.items():
    score = 0
    for en, ref in zip(ens, refs):
        pred = translate_long_text(en, model, tokenizer, device)
        score += compute_bleu(pred, ref)
    results[name] = score/len(refs)

base_score = results.get("Base", 0)
print("\n" + "="*65)
print(f"{'Модель':<35} | {'SacreBLEU':>10} | {'Diff':>10}")
print("-" * 65)
for name, score in results.items():
    diff = f"{score - base_score:+.2f}" if name != "Base" else "baseline"
    print(f"{name:<35} | {score:>10.2f} | {diff:>10}")
print("="*65)


Модель                              |  SacreBLEU |       Diff
-----------------------------------------------------------------
Base                                |      10.20 |   baseline
Postpunk                            |       1.26 |      -8.94
Ru_en_story_pairs                   |       3.32 |      -6.88
Go_ru_en                            |       3.34 |      -6.86
Wikihow                             |       6.42 |      -3.78


In [32]:
n = 2
print(f"EN:  {ens[n]}")
print(f"RU: {refs[n]}")
print('-'*60)

for name, model in loaded_models.items():
    pred = translate_long_text(ens[n], model, tokenizer, device)

    print(f"{name}: {pred}")
    bleu = compute_bleu(pred, [refs[n]])
    print(f"{name} BLEU: {bleu:.2f}")
    print('-'*60)

EN:  Room design is different than decorating because it has to do with the whole space, including the parts of it that are permanent like the walls, windows, and floors. When you begin your project, you need to get everything else out of the way so that you can see the bare bones of what you have to work with.  Begin by moving furniture and every item of décor (including pictures on the walls) out of the room. Keep it in another room if you can, to give you time to finish your project before you decide what to give away or sell. Give the space a deep cleaning. Clean the walls, windows, and floors and any permanent fixtures like lights, light switches, cabinets, or baseboards. In order to prevent getting paint or adhesive on the new floors, finish the walls before doing any work to replace flooring.  You may need to remove old wall paper or old wood trim before beginning. Prime the walls and paint the walls and trim. If you plan to replace carpeting, vinyl, tile, or wood floors, you sh